# Palm92 RepoGuard v0.9: medium-v002 follow-up

The diagnostic-v002 experiment repaired the invoice task in four actions with automatic test verification. This follow-up runs the **same three medium tasks** as archived medium-v001 with the updated loop guard. These are small authored development tasks, not held-out evaluation. Do not compare their overall rate with the easier v0.7/v0.8 task pack. Choose a T4 GPU first.

## Step 1: Prepare a clean checkout of the v0.9 branch

Start in `/content` before deleting any old temporary clone. This notebook never deletes saved evidence in Google Drive.

In [ ]:
%cd /content
!rm -rf /content/palm92-repoguard
!git clone --branch v0.9-hard-benchmarks --single-branch https://github.com/faithfulord1/palm92-repoguard.git /content/palm92-repoguard
%cd /content/palm92-repoguard
!python -m pip install -e '.[gemma,dev]'

## Step 2: Run RepoGuard regression tests and confirm the three fixture defects

The fixture checker is expected to see failing tests **before** the agent fixes those isolated copies. If it reports collection errors or fails its validation, stop.

In [ ]:
!python -m pytest -q
!python scripts/check_v0_9_fixtures.py

## Step 3: Confirm GPU

Stop if `CUDA available: False`. Check Runtime > Change runtime type > T4 GPU, then rerun setup if the runtime changes.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA GPU available: stop before the benchmark.'
print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(i, p.name, round(p.total_memory / (1024**3), 1), 'GiB')

## Step 4: Run medium-v002

This run is separate from archived medium-v001. If you have already run medium-v002 in this runtime, stop and choose a new experiment ID; the runner will not overwrite existing evidence.

In [ ]:
%cd /content/palm92-repoguard
!python scripts/run_v0_9_benchmark.py --experiment-id medium-v002

## Step 5: Inspect the summary and detailed JSONL

A reported fix must be checked against actual post-write test results. These small public-test tasks are not held-out evaluation.

In [ ]:
%cd /content/palm92-repoguard
from pathlib import Path
p = Path('artifacts/v0.9/medium-v002')
for name in ('summary.json', 'tasks.jsonl'):
    f = p / name
    print('\nFILE', f, 'exists:', f.exists())
    if f.exists():
        print(f.read_text()[:30000])

## Step 6: Download evidence before ending Colab

Save the whole new v0.9 run folder off the ephemeral runtime. The locked v0.7 and v0.8 artifacts must not be overwritten.

In [ ]:
%cd /content/palm92-repoguard
from google.colab import files
from pathlib import Path
import shutil
folder = Path('artifacts/v0.9/medium-v002')
assert (folder / 'summary.json').exists() and (folder / 'tasks.jsonl').exists(), 'Missing run evidence; finish Step 4 first.'
archive = shutil.make_archive('/content/repoguard-v0.9-medium-v002', 'zip', root_dir=folder)
files.download(archive)